# Nemotron Reasoning Challenge — Kaggle workflow

Runs the repo pipeline on a **Kaggle GPU notebook** or **locally** (e.g. **RTX PRO 6000**), aligned with the competition setup:

- **Data**: e.g. `/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv` (mount varies).
- **Code dataset**: Kaggle Dataset with latest `scripts/`.
- **Offline packages (optional)**: [dennisfong/nvidia-nemotron-offline-packages](https://www.kaggle.com/datasets/dennisfong/nvidia-nemotron-offline-packages) — add as input; the install cell uses `--find-links` (still falls back to PyPI for missing wheels).
- **Base weights**: competition **Model** under `/kaggle/input/models/...` or `kagglehub` / HF id.
- **GPU profile**: `GPU_PROFILE = "t4"` vs `"high_vram"` (bf16 + longer seq for RTX PRO 6000 / A100-class).

**Before you run:** competition + scripts dataset + (optional) offline wheel dataset + Model input.

**Outputs** in `WORK_ROOT`: `data/`, `lora_adapter/`, `submission.zip`.


In [ ]:
# --- Configuration ---
from pathlib import Path
import os
import shutil

IS_KAGGLE = os.path.exists("/kaggle/input")

WORK_ROOT = Path("/kaggle/working/project" if IS_KAGGLE else ".").resolve()

KAGGLE_COMPETITION_SLUG = "nvidia-nemotron-model-reasoning-challenge"

KAGGLE_MODEL_HUB_SLUG = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
USE_KAGGLEHUB_MODEL = True

CODE_SOURCE = "/kaggle/input/datasets/sebmontreal/nvidia-nemotron-reasoning-challenge-source"
COMPETITION_DATA_DIR = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"

SKIP_COT = True
SYNTHETIC_PER_KIND = 400
RUN_VLLM_EVAL = False

LORA_TARGET_MODE = "kaggle_nemotron"
LORA_ALPHA = 16

TRAIN_BATCH = 1
GRAD_ACCUM = 16
NUM_EPOCHS = 2.0

# --- GPU profile ---
# "t4"        — Kaggle T4 ~16GB: 4-bit QLoRA, ctx 2048
# "high_vram" — RTX PRO 6000 / A100 / H100: bf16 --no-quant, longer ctx
GPU_PROFILE = "t4" if IS_KAGGLE else "high_vram"
# Force profile, e.g. on a big-GPU VM: GPU_PROFILE = "high_vram"

if GPU_PROFILE == "high_vram":
    USE_BF16_FULL = True
    TRAIN_MAX_SEQ = 4096
    TRAIN_MAX_MEMORY_JSON = None
elif GPU_PROFILE == "t4":
    USE_BF16_FULL = False
    TRAIN_MAX_SEQ = 2048
    TRAIN_MAX_MEMORY_JSON = None
else:
    raise ValueError(f"Unknown GPU_PROFILE: {GPU_PROFILE!r}")

# --- Offline wheels: dennisfong/nvidia-nemotron-offline-packages ---
USE_OFFLINE_WHEELS = True
OFFLINE_WHEELS_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages"

# True: pip uses only --find-links (no PyPI / DNS). Use when you see "name resolution" errors.
PIP_NO_INDEX = True
# If PyPI works, set PIP_NO_INDEX = False so missing wheels can be pulled from the internet.

MODEL_PATH_LOCAL = "nvidia/Nemotron-3-Nano-30B-A3B-BF16"

print("IS_KAGGLE:", IS_KAGGLE)
print("WORK_ROOT:", WORK_ROOT)
print("GPU_PROFILE:", GPU_PROFILE, "| USE_BF16_FULL:", USE_BF16_FULL, "| TRAIN_MAX_SEQ:", TRAIN_MAX_SEQ)
print("USE_OFFLINE_WHEELS:", USE_OFFLINE_WHEELS)
print("PIP_NO_INDEX:", PIP_NO_INDEX)


## Install dependencies

Wheels come from **`OFFLINE_WHEELS_DIR`** via **`--find-links`**. **`PIP_NO_INDEX = True`** adds **`--no-index`**, so pip **never** contacts PyPI (fixes **`Temporary failure in name resolution`**). Every package you need must have a **`.whl`** in that folder (or pip will fail).

If **`kagglehub`** is not in the bundle, the install cell skips it — use the competition **Model** mount only and set **`USE_KAGGLEHUB_MODEL = False`** in the config cell.

Kaggle ships **`torch`**. **Nemotron-3** needs **`mamba-ssm`** and **`causal-conv1d`**. Prefer **`transformers>=4.45,<5`** on small GPUs.

**RTX PRO 6000 / local:** **`GPU_PROFILE = "high_vram"`**; install CUDA **`torch`** before this notebook if needed.


In [ ]:
import subprocess
import sys
from pathlib import Path


def _wheel_index_dir(base: Path) -> Path | None:
    if not base.is_dir():
        return None
    if list(base.glob("*.whl")):
        return base
    for sub in ("wheels", "wheel", "packages", "pip"):
        d = base / sub
        if d.is_dir() and list(d.glob("*.whl")):
            return d
    if list(base.rglob("*.whl")):
        return base
    return None


def find_offline_wheel_root() -> Path | None:
    if not USE_OFFLINE_WHEELS:
        return None
    if OFFLINE_WHEELS_DIR:
        p = Path(OFFLINE_WHEELS_DIR)
        idx = _wheel_index_dir(p)
        if idx:
            return idx
    if IS_KAGGLE:
        root = Path("/kaggle/input")
        if root.is_dir():
            for name in (
                "datasets/dennisfong/nvidia-nemotron-offline-packages",
                "nvidia-nemotron-offline-packages",
                "dennisfong-nvidia-nemotron-offline-packages",
            ):
                idx = _wheel_index_dir(root / name)
                if idx:
                    return idx
            best, best_n = None, 0
            for p in root.iterdir():
                if not p.is_dir():
                    continue
                idx = _wheel_index_dir(p)
                if idx is None:
                    continue
                n = len(list(idx.rglob("*.whl")))
                if n > best_n:
                    best_n, best = n, idx
            if best is not None and best_n >= 2:
                return best
    if OFFLINE_WHEELS_DIR:
        return _wheel_index_dir(Path(OFFLINE_WHEELS_DIR))
    return None


def pip_install(find_links: list[str], packages: list[str], *, no_index: bool = False) -> None:
    extra = ["--no-index"] if no_index else []
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *extra, *find_links, *packages]
    subprocess.run(cmd, check=True)


_wheel_root = find_offline_wheel_root()
_fl: list[str] = []
if _wheel_root is not None:
    _fl = [f"--find-links={_wheel_root.resolve().as_uri()}"]
    _n = len(list(_wheel_root.rglob("*.whl")))
    print("Using offline find-links:", _wheel_root, f"({_n} wheels)")
elif USE_OFFLINE_WHEELS and IS_KAGGLE:
    print("WARNING: USE_OFFLINE_WHEELS but no wheel folder found — using PyPI only.")

_no_idx = bool(PIP_NO_INDEX and _wheel_root is not None)
if PIP_NO_INDEX and _wheel_root is None:
    raise SystemExit(
        "PIP_NO_INDEX is True but no wheel directory found. Fix OFFLINE_WHEELS_DIR or set PIP_NO_INDEX=False."
    )

if _no_idx:
    print("pip: --no-index (no PyPI / DNS)")

if not _no_idx:
    pip_install(_fl, ["kagglehub"], no_index=False)
else:
    try:
        pip_install(_fl, ["kagglehub"], no_index=True)
    except subprocess.CalledProcessError:
        print(
            "WARN: kagglehub not found in offline wheels. Set USE_KAGGLEHUB_MODEL=False in config "
            "and use the competition Model input for base weights."
        )

_core = [
    "transformers>=4.45,<5",
    "peft",
    "trl",
    "datasets",
    "psutil",
    "accelerate",
    "bitsandbytes",
    "pandas",
    "polars",
    "tqdm",
    "scikit-learn",
    "huggingface_hub",
]
pip_install(_fl, _core, no_index=_no_idx)
pip_install(_fl, ["causal-conv1d", "mamba-ssm"], no_index=_no_idx)
try:
    pip_install(_fl, ["unsloth"], no_index=_no_idx)
except subprocess.CalledProcessError:
    print("unsloth install skipped (optional).")

if False:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)


## Bootstrap: copy `scripts/` + competition CSVs into `WORK_ROOT`

In [ ]:
def find_subdir_with_csv(name: str) -> Path | None:
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    for p in sorted(root.iterdir()):
        if p.is_dir() and (p / name).is_file():
            return p
    return None


WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)

_kaggle_input_root = Path("/kaggle/input")
if CODE_SOURCE is None and IS_KAGGLE and _kaggle_input_root.is_dir():
    for candidate in _kaggle_input_root.iterdir():
        if (candidate / "scripts" / "01_eda.py").is_file():
            CODE_SOURCE = candidate
            break

if CODE_SOURCE is not None:
    CODE_SOURCE = Path(CODE_SOURCE)
    for item in ["scripts", "requirements.txt", "requirements-vllm.txt", "README.md"]:
        src = CODE_SOURCE / item
        if not src.exists():
            continue
        dst = WORK_ROOT / item
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print("Copied code from", CODE_SOURCE)
else:
    if not (WORK_ROOT / "scripts" / "01_eda.py").is_file():
        raise FileNotFoundError(
            "Could not find scripts/. Set CODE_SOURCE to a Kaggle Dataset containing this repo."
        )

if COMPETITION_DATA_DIR is None and IS_KAGGLE:
    _comp_cands = [
        Path("/kaggle/input/competitions") / KAGGLE_COMPETITION_SLUG,
        Path("/kaggle/input") / KAGGLE_COMPETITION_SLUG,
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge"),
    ]
    for preferred in _comp_cands:
        if preferred.is_dir() and (preferred / "train.csv").is_file():
            COMPETITION_DATA_DIR = preferred
            print("Using competition path:", COMPETITION_DATA_DIR)
            break

if COMPETITION_DATA_DIR is None:
    COMPETITION_DATA_DIR = find_subdir_with_csv("train.csv")

if COMPETITION_DATA_DIR is None:
    raise FileNotFoundError(
        "No train.csv under /kaggle/input. Add the competition or set COMPETITION_DATA_DIR."
    )

COMPETITION_DATA_DIR = Path(COMPETITION_DATA_DIR)
data_dir = WORK_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)
for fname in ("train.csv", "test.csv"):
    src = COMPETITION_DATA_DIR / fname
    if src.is_file():
        shutil.copy2(src, data_dir / fname)
        print("Copied", fname)
    else:
        print("Missing (optional):", src)

import sys

sys.path.insert(0, str(WORK_ROOT))
print("cwd:", os.getcwd())

## Download base model (competition mount → Kaggle Hub → HF)

1. If you add the competition **Model** input, weights appear under `/kaggle/input/models/...` (no download).
2. Else `kagglehub.model_download(...)` when `USE_KAGGLEHUB_MODEL` is True.
3. Else Hugging Face id `nvidia/Nemotron-3-Nano-30B-A3B-BF16` (needs `HF_TOKEN` if gated).

`03_train_lora.py` also resolves the same mount or `NEMOTRON_MODEL_PATH` when `--model-path` is still the HF id.

In [ ]:
from pathlib import Path


def _find_competition_nemotron_weights() -> str | None:
    preferred = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16")
    for root in (preferred, Path("/kaggle/input/models")):
        if not root.is_dir():
            continue
        configs = [c for c in root.rglob("config.json") if c.is_file()]
        if not configs:
            continue
        if root == preferred:
            configs.sort(key=lambda p: len(p.parts))
            return str(configs[0].parent.resolve())
        for cfg in sorted(configs, key=lambda p: len(str(p))):
            low = str(cfg).lower()
            if "nemotron" in low and "nano" in low:
                return str(cfg.parent.resolve())
    return None


_env_weights = os.environ.get("NEMOTRON_MODEL_PATH", "").strip()
if _env_weights and Path(_env_weights).is_dir() and (Path(_env_weights) / "config.json").is_file():
    MODEL_PATH_LOCAL = str(Path(_env_weights).resolve())
    print("Using NEMOTRON_MODEL_PATH:", MODEL_PATH_LOCAL)
elif IS_KAGGLE:
    _comp = _find_competition_nemotron_weights()
    if _comp:
        MODEL_PATH_LOCAL = _comp
        print("Using competition Model mount:", MODEL_PATH_LOCAL)
    elif USE_KAGGLEHUB_MODEL:
        import kagglehub

        MODEL_PATH_LOCAL = kagglehub.model_download(KAGGLE_MODEL_HUB_SLUG)
        print("Model from kagglehub:", MODEL_PATH_LOCAL)
    else:
        MODEL_PATH_LOCAL = "nvidia/Nemotron-3-Nano-30B-A3B-BF16"
        print("Using HF id:", MODEL_PATH_LOCAL)
else:
    print("MODEL_PATH_LOCAL (from config):", MODEL_PATH_LOCAL)

## Hugging Face login (only if you load from HF)

Skip if `USE_KAGGLEHUB_MODEL` is True and you only use the Kaggle Hub path above.

In [ ]:
from huggingface_hub import login

# Put the short branch first so a partial copy/paste cannot leave a bare `if` without a body.
if IS_KAGGLE and USE_KAGGLEHUB_MODEL:
    print("Skipping HF login (using Kaggle Hub weights).")
else:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False)
        print("HF login: OK (env token)")
    else:
        try:
            login(add_to_git_credential=False)
            print("HF login: OK")
        except Exception as e:
            print("HF login:", e)

## Peek `train.csv` (Polars, optional)

In [ ]:
import polars as pl

train_pl = pl.read_csv(WORK_ROOT / "data" / "train.csv")
print(train_pl.head())
print(train_pl.shape)

## Phase 1 — EDA

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/01_eda.py",
        "--data-dir",
        "data",
        "--report-dir",
        "data/reports",
        "--tokenizer-model",
        str(MODEL_PATH_LOCAL),
    ],
    check=True,
)

## Phase 2 — Prepare SFT JSONL

In [ ]:
import sys

skip = ["--skip-cot"] if SKIP_COT else []
cmd = [
    sys.executable,
    "scripts/02_prepare_data.py",
    "--data-dir",
    "data",
    "--synthetic-dir",
    "data/synthetic",
    "--output",
    "data/train_sft.jsonl",
    "--tokenizer-model",
    str(MODEL_PATH_LOCAL),
    "--synthetic-per-kind",
    str(SYNTHETIC_PER_KIND),
] + skip
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Phase 3 — LoRA training

**`GPU_PROFILE == "t4"`:** 4-bit QLoRA (`USE_BF16_FULL = False`), **`transformers>=4.45,<5`**. Auto `max_memory` + RAM-safe `cpu` cap (see `03_train_lora.py`).

**`GPU_PROFILE == "high_vram"` (RTX PRO 6000, etc.):** bf16 + **`--no-quant`**, longer **`TRAIN_MAX_SEQ`**. Ensure **`torch`** matches your CUDA build locally.

Logs: **`[03_train_lora] build=...`**, **`offload_folder=...`**. First step can take several minutes.


In [ ]:
import os
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

_offload = str((WORK_ROOT / "lora_output" / "hf_offload").resolve())
cmd = [
    sys.executable,
    "scripts/03_train_lora.py",
    "--data-path",
    "data/train_sft.jsonl",
    "--output-dir",
    "lora_adapter",
    "--checkpoint-dir",
    "lora_output",
    "--offload-folder",
    _offload,
    "--model-path",
    str(MODEL_PATH_LOCAL),
    "--lora-target-mode",
    LORA_TARGET_MODE,
    "--lora-alpha",
    str(LORA_ALPHA),
    "--batch-size",
    str(TRAIN_BATCH),
    "--grad-accum",
    str(GRAD_ACCUM),
    "--epochs",
    str(NUM_EPOCHS),
    "--max-seq-length",
    str(TRAIN_MAX_SEQ),
    "--force-peft",
    "--dataloader-workers",
    "0",
]
if USE_BF16_FULL:
    cmd.insert(cmd.index("--lora-target-mode"), "--no-quant")
if TRAIN_MAX_MEMORY_JSON:
    cmd.extend(["--max-memory-json", TRAIN_MAX_MEMORY_JSON])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Phase 4 — vLLM evaluation (optional)

In [ ]:
import subprocess
import sys

if RUN_VLLM_EVAL:
    subprocess.run(
        [
            sys.executable,
            "scripts/04_evaluate.py",
            "--adapter-path",
            "lora_adapter",
            "--data-dir",
            "data",
            "--max-samples",
            "32",
        ],
        check=True,
    )
else:
    print("Skipping vLLM eval.")

## Phase 5 — `submission.zip`

Uses `scripts/05_package_submission.py` (adapter-only zip). The competition template sometimes uses `zip submission.zip adapter_*` from `/kaggle/working`; this keeps only LoRA artifacts.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/05_package_submission.py",
        "--adapter-dir",
        "lora_adapter",
        "--output",
        "submission.zip",
    ],
    check=True,
)

zp = WORK_ROOT / "submission.zip"
print("submission.zip:", zp.is_file(), zp.stat().st_size if zp.is_file() else 0)

if IS_KAGGLE:
    from IPython.display import FileLink, display

    display(FileLink("submission.zip"))